# EPSM GeoJSON Processing Pipeline Demo

This notebook demonstrates how to use the EPSM pipeline to:
1. Download building footprints from DTCC (optional - requires DTCC library)
2. Process local GeoJSON files
3. Convert GeoJSON to EnergyPlus IDF files using Ladybug Tools

**Authors:** Aaron Qiyu Liu and Sanjay Somanath

In [ ]:
# Import required modules
import sys
from pathlib import Path

# Add parent directory to path to import geojson_processor
sys.path.insert(0, str(Path.cwd().parent))

from geojson_processor.geojson_to_idf import GeoJSONToIDFConverter
from geojson_processor.dtcc_service import DTCCService

## Option 1: Download Building Data from DTCC (Requires DTCC Library)

If you have the DTCC library installed, you can download building footprints directly from DTCC's database for any area in Sweden.

In [ ]:
# Example: Download building data from DTCC for a specific area in Gothenburg
# Note: This requires the DTCC library to be installed

# Create working directory
work_dir = Path('dtcc_output')
work_dir.mkdir(exist_ok=True)

# Initialize DTCC service
dtcc = DTCCService(str(work_dir))

# Define bounds for area in Gothenburg (EPSG:3006 - Swedish coordinate system)
bounds = {
    'west': 318000,   # Western boundary
    'south': 6399000,  # Southern boundary
    'east': 318500,    # Eastern boundary
    'north': 6399500   # Northern boundary
}

# Download city data (building footprints and terrain)
try:
    geojson_path, terrain_path = dtcc.download_city_data(
        west=bounds['west'],
        south=bounds['south'],
        east=bounds['east'],
        north=bounds['north'],
        epsg=3006
    )
    print(f"✅ Downloaded GeoJSON: {geojson_path}")
    print(f"✅ Terrain STL: {terrain_path}")
except Exception as e:
    print(f"❌ DTCC download failed: {e}")
    print("Note: DTCC library may not be installed. See Option 2 for local GeoJSON processing.")

DTCC library not available: No module named 'dtcc_core'


❌ DTCC download failed: dtcc library is required. Install with: pip install dtcc
Note: DTCC library may not be installed. See Option 2 for local GeoJSON processing.


## Option 2: Process Local GeoJSON File (Recommended - No DTCC Required)

This is the main workflow that converts GeoJSON building footprints to EnergyPlus IDF files.
You can use any GeoJSON file with building footprints.

In [ ]:
# Step 1: Initialize converter with a working directory
output_dir = Path('idf_output')
output_dir.mkdir(exist_ok=True)

converter = GeoJSONToIDFConverter(str(output_dir))
print(f"✅ Converter initialized with working directory: {output_dir}")

In [ ]:
# Step 2: Process your GeoJSON file
# Replace 'your_buildings.geojson' with your actual GeoJSON file path

# Path to Uddevala.geojson (in parent directory)
input_geojson = Path('../../Uddevala.geojson')  # Relative path from notebooks folder

# Option A: Full pipeline (filter → enrich → convert to IDF)
if input_geojson.exists():
    idf_path, gbxml_path = converter.process(
        geojson_path=input_geojson,
        filter_height_min=3,          # Minimum building height (meters)
        filter_height_max=100,        # Maximum building height (meters)
        filter_area_min=100,          # Minimum building area (m²)
        use_multiplier=False,         # Use floor multipliers for faster simulation
        simulation_bounds=None,       # Optional: dict with 'north', 'south', 'east', 'west'
        timestep=1,                   # Timesteps per hour (1=fast, 6=detailed)
        do_zone_sizing=False,         # HVAC sizing calculations (False=faster)
        do_system_sizing=False,       # System sizing calculations (False=faster)
        winter_design_temp=-10,       # Winter design temperature (°C)
        summer_design_temp=30         # Summer design temperature (°C)
    )
    
    print(f"✅ IDF file generated: {idf_path}")
    if gbxml_path:
        print(f"✅ GBXML file generated: {gbxml_path}")
else:
    print(f"❌ GeoJSON file not found: {input_geojson}")
    print("Please provide a valid GeoJSON file with building footprints.")


### Option 2B: Step-by-step Processing

If you want more control, you can run each step separately:

In [ ]:
# Step-by-step processing example

if input_geojson.exists():
    # Step 1: Filter buildings by size
    filtered_geojson = converter.filter_buildings(
        geojson_path=input_geojson,
        filter_height_less_than=3,      # Min height
        filter_height_greater_than=100, # Max height
        filter_area_less_than=100       # Min area
    )
    print(f"✅ Filtered GeoJSON: {filtered_geojson}")
    
    # Step 2: Enrich with metadata (required for Dragonfly)
    enriched_geojson = converter.enrich_geojson(
        geojson_path=filtered_geojson,
        simulation_bounds=None  # Optional: specify bounds to mark buildings as simulated vs context
    )
    print(f"✅ Enriched GeoJSON: {enriched_geojson}")
    
    # Step 3: Convert to IDF
    idf_path, gbxml_path = converter.convert_to_idf(
        geojson_path=enriched_geojson,
        use_multiplier=False,
        timestep=1,
        do_zone_sizing=False,
        do_system_sizing=False,
        winter_design_temp=-10,
        summer_design_temp=30
    )
    print(f"✅ Final IDF: {idf_path}")

## Advanced: Specify Simulation Bounds

You can mark specific buildings for simulation while keeping others as context shading:

In [ ]:
# Define simulation bounds (in EPSG:3006 for Swedish coordinates)
# Buildings inside will be simulated, outside will be context shading only
simulation_bounds = {
    'north': 6399500,
    'south': 6399000,
    'east': 318500,
    'west': 318000
}

# Process with simulation bounds
if input_geojson.exists():
    idf_path, _ = converter.process(
        geojson_path=input_geojson,
        simulation_bounds=simulation_bounds,
        filter_height_min=3,
        filter_height_max=100,
        filter_area_min=100
    )
    print(f"✅ Generated IDF with simulation bounds: {idf_path}")

## Summary: Key Parameters

### GeoJSON Requirements
- Must contain building footprints as Polygons or MultiPolygons
- Should include `height` property (meters)
- Coordinate system: EPSG:4326 (WGS84) or will be converted

### Important Parameters

**Filtering:**
- `filter_height_min`: Minimum building height (default: 3m)
- `filter_height_max`: Maximum building height (default: 100m)
- `filter_area_min`: Minimum building footprint area (default: 100m²)

**Simulation Speed:**
- `timestep`: 1 = fast (hourly), 6 = detailed (10-min intervals)
- `use_multiplier`: True = faster (use floor multipliers)
- `do_zone_sizing`: False = faster (skip HVAC sizing)
- `do_system_sizing`: False = faster (skip system sizing)

**Design Conditions:**
- `winter_design_temp`: Winter design temperature in °C (default: -10)
- `summer_design_temp`: Summer design temperature in °C (default: 30)

**Simulation Bounds:**
- Dictionary with 'north', 'south', 'east', 'west' in EPSG:3006
- Buildings inside = simulated, outside = context shading only